In [175]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Embedding, Dropout, LayerNormalization, TextVectorization
from tensorflow.keras.models import Model
import numpy as np

## Defining Positional Encoding

In [176]:
def positional_encoding(position, d_model):
    angle_rads = np.arange(position)[:, np.newaxis] / np.power(
        10000, (2 * (np.arange(d_model) // 2)) / np.float32(d_model)
    )
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])  # apply sin to even indices
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])  # apply cos to odd indices
    pos_encoding = angle_rads[np.newaxis, ...]
    return tf.cast(pos_encoding, dtype=tf.float32)

## Defining Multi-head Attention 

In [177]:
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)
        self.dense = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def scaled_dot_product_attention(self, q, k, v, mask):
              matmul_qk = tf.matmul(q, k, transpose_b=True)
              dk = tf.cast(tf.shape(k)[-1], tf.float32)
              scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
          
              if mask is not None:
                  scaled_attention_logits += (mask * -1e9)
          
              attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
              output = tf.matmul(attention_weights, v)
              return output, attention_weights
    
    def call(self, v, k, q, mask):
                batch_size = tf.shape(q)[0]
                q = self.wq(q)
                k = self.wk(k)
                v = self.wv(v)
                q = self.split_heads(q, batch_size)
                k = self.split_heads(k, batch_size)
                v = self.split_heads(v, batch_size)
                
                attention, attention_weights = self.scaled_dot_product_attention(q, k, v, mask)
                attention = tf.transpose(attention, perm=[0, 2, 1, 3])
                attention = tf.reshape(attention, (batch_size, -1, self.d_model))
                output = self.dense(attention)
                return output

## Defining Feed Forward Network


In [178]:
class PositionwiseFeedforward(tf.keras.layers.Layer):
    def __init__(self, d_model, dff):
        super().__init__()
        self.dense1 = Dense(dff, activation='relu')
        self.dense2 = Dense(d_model)

    def call(self, x):
        x = self.dense1(x)
        return self.dense2(x)

## Defining Transformer Block


In [179]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.att = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedforward(d_model, dff)
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, x, training=False, mask=None):
        attn_output = self.att(x, x, x, mask=mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

## Encoder

In [180]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size, maximum_position_encoding, dropout_rate=0.1):
        super(Encoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        
        self.embedding = Embedding(input_vocab_size, d_model)
        self.pos_encoding = positional_encoding(maximum_position_encoding, d_model)
        self.dropout = Dropout(dropout_rate)
        self.enc_layers = [TransformerBlock(
            d_model, num_heads, dff, dropout_rate) for _ in range(num_layers)
            ]

    def call(self, x, training, mask):
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        # for i in range(self.num_layers):
        #     x = self.enc_layers[i](x, training=training, mask=mask)
        for layer in self.enc_layers:
            x = layer(x, training=training, mask=mask)
            
        return x

## Decoder Block

In [181]:
class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        
        self.mha2 = MultiHeadAttention(d_model, num_heads)

        self.ffn = PositionwiseFeedforward(d_model, dff)

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)

        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
        self.dropout3 = Dropout(dropout_rate)

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        attn1 = self.mha1(x, x, x, mask=look_ahead_mask) 
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(attn1 + x)

        attn2 = self.mha2(v=enc_output, k=enc_output, q=out1, mask=padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(attn2 + out1)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(ffn_output + out2)

        return out3

## Decoder

In [182]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size, maximum_position_encoding, dropout_rate=0.1):
        super(Decoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = Embedding(target_vocab_size, d_model)
        self.pos_encoding = positional_encoding(
            maximum_position_encoding, d_model)
        self.dropout = Dropout(dropout_rate)
        self.dec_layers = [DecoderBlock(
            d_model, num_heads, dff, dropout_rate) for _ in range(num_layers)]

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        seq_len = tf.shape(x)[1]
        attention_weights = {}
        x = self.embedding(x)
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        # for i in range(self.num_layers):
        #     x = self.dec_layers[i](x, training=training, mask=look_ahead_mask)
        for layer in self.dec_layers:
            x = layer(x, enc_output=enc_output, training=training, look_ahead_mask=look_ahead_mask, padding_mask=padding_mask)
        return x, attention_weights

## Defining Transformer Model

In [183]:
class Transformer(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff,
                 input_vocab_size, target_vocab_size, maximum_position_encoding, dropout_rate=0.1):
        super(Transformer, self).__init__()
        self.encoder = Encoder(
            num_layers, 
            d_model, 
            num_heads, 
            dff,
            input_vocab_size, 
            maximum_position_encoding, 
            dropout_rate
        )
        self.decoder = Decoder(
            num_layers, 
            d_model, 
            num_heads, 
            dff,
            target_vocab_size, 
            maximum_position_encoding, 
            dropout_rate
        )
        self.final_layer = Dense(target_vocab_size)

    def call(self, inputs, training=False, look_ahead_mask=None, padding_mask=None):
        inp, tar = inputs
        enc_output = self.encoder(
            inp, 
            training=training, 
            mask=padding_mask
            )
        dec_output, _ = self.decoder(
            tar, 
            enc_output=enc_output, 
            training=training,
            look_ahead_mask=look_ahead_mask, 
            padding_mask=padding_mask
        )
        final_output = self.final_layer(dec_output)

        return final_output

## Training and testing the Model

In [184]:
# Defining Custom Parameters
num_layers = 4
d_model = 128
num_heads = 8
dff = 512
input_vocab_size = 8500
target_vocab_size = 8000
maximum_position_encoding = 10000
dropout_rate = 0.1

transformer = Transformer(
    num_layers,
    d_model,
    num_heads,
    dff,
    input_vocab_size,
    target_vocab_size,
    maximum_position_encoding,
    dropout_rate
)

inputs = tf.random.uniform(
    (64, 50), dtype=tf.int64, minval=0, maxval=input_vocab_size
    )
targets = tf.random.uniform(
    (64, 50), dtype=tf.int64, minval=0, maxval=target_vocab_size
    )

look_ahead_mask = None
padding_mask = None

output = transformer((inputs, targets), training=True,
                     look_ahead_mask=look_ahead_mask, padding_mask=padding_mask)
print(output.shape)

(64, 50, 8000)


## Translation Test

In [185]:
def create_padding_mask(seq):
    seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
    return seq[:, tf.newaxis, tf.newaxis, :]  # (batch_size, 1, 1, seq_len)

def create_look_ahead_mask(size):
    mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    return mask  # (seq_len, seq_len)

# --- TEST CONFIGURATION ---
batch_size = 1
seq_len_in = 5   # e.g., "I love deep learning <pad>"
seq_len_out = 6  # e.g., "<start> I love deep learning <pad>"

# source: [12, 45, 7, 102, 0] 
sample_encoder_input = tf.constant([[12, 45, 7, 102, 0]], dtype=tf.int64)
# target: [1, 54, 89, 210, 22, 0]
sample_decoder_input = tf.constant([[1, 54, 89, 210, 22, 0]], dtype=tf.int64)

# Create masks
enc_padding_mask = create_padding_mask(sample_encoder_input)

look_ahead_mask = create_look_ahead_mask(tf.shape(sample_decoder_input)[1])
dec_padding_mask = create_padding_mask(sample_decoder_input)
combined_mask = tf.maximum(dec_padding_mask, look_ahead_mask)

# Run the inference
predictions = transformer(
    inputs=(sample_encoder_input, sample_decoder_input),
    training=False,
    look_ahead_mask=combined_mask,
    padding_mask=enc_padding_mask
)

print(f"Encoder Input (English) shape: {sample_encoder_input.shape}")
print(f"Decoder Input (French) shape: {sample_decoder_input.shape}")
print("-" * 30)
print(f"Transformer Output shape: {predictions.shape}") 
# Expected: (1, 6, 8000) -> (Batch, Target_Seq_Len, Target_Vocab_Size)

# Extract the most likely token IDs
predicted_ids = tf.argmax(predictions, axis=-1)
print(f"Predicted IDs for the translation: {predicted_ids.numpy()}")

Encoder Input (English) shape: (1, 5)
Decoder Input (French) shape: (1, 6)
------------------------------
Transformer Output shape: (1, 6, 8000)
Predicted IDs for the translation: [[2981 2981 2981 2981 2981 2981]]


In [186]:
# 1. Create a dummy vocabulary to match your 8000-word limit
# This maps every possible ID (0-7999) to a placeholder string
vocabulary = [f"token_{i}" for i in range(target_vocab_size)]

# 2. Get the IDs from your result
ids = predicted_ids.numpy()[0]

# 3. Convert IDs to "Text"
translated_text = [vocabulary[i] for i in ids]

print("-" * 30)
print(f"Predicted IDs: {ids}")
print(f"Translated Text: {' '.join(translated_text)}")

------------------------------
Predicted IDs: [2981 2981 2981 2981 2981 2981]
Translated Text: token_2981 token_2981 token_2981 token_2981 token_2981 token_2981


## Loss and Optimizer

In [187]:
# Loss Function
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
    # Mask to ignore padding (0) in the loss calculation
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_sum(loss_)/tf.reduce_sum(mask)

# Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

train_loss = tf.keras.metrics.Mean(name='train_loss')

## Training

In [188]:
@tf.function
def train_step(inp, tar):
    # Target input: <start> Je suis ... (exclude last word)
    # Target real: Je suis étudiant <end> (exclude first word)
    tar_inp = tar[:, :-1]
    tar_real = tar[:, 1:]

    # Create masks
    enc_padding_mask = create_padding_mask(inp)
    look_ahead_mask = create_look_ahead_mask(tf.shape(tar_inp)[1])
    dec_padding_mask = create_padding_mask(tar_inp)
    combined_mask = tf.maximum(dec_padding_mask, look_ahead_mask)

    with tf.GradientTape() as tape:
        predictions = transformer(
            (inp, tar_inp), 
            training=True,
            look_ahead_mask=combined_mask,
            padding_mask=enc_padding_mask
        )
        loss = loss_function(tar_real, predictions)

    gradients = tape.gradient(loss, transformer.trainable_variables)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))

    train_loss(loss)

In [189]:
import time

# Toy Data: "I am a student" -> "<start> Je suis étudiant <end>"
# Let's assume these are the IDs after tokenization
toy_inp = tf.constant([[12, 45, 7, 102, 0]], dtype=tf.int64)
toy_tar = tf.constant([[1, 54, 89, 210, 2, 0]], dtype=tf.int64)

EPOCHS = 1000

for epoch in range(EPOCHS):
    start = time.time()
    train_loss.reset_state()

    train_step(toy_inp, toy_tar)

    print(f'Epoch {epoch + 1} Loss {train_loss.result():.4f} Time {time.time() - start:.2f}s')

Epoch 1 Loss 9.0411 Time 6.05s
Epoch 2 Loss 8.8467 Time 0.01s
Epoch 3 Loss 8.7188 Time 0.01s
Epoch 4 Loss 8.6116 Time 0.01s
Epoch 5 Loss 8.5563 Time 0.01s
Epoch 6 Loss 8.4222 Time 0.01s
Epoch 7 Loss 8.4137 Time 0.01s
Epoch 8 Loss 8.3351 Time 0.01s
Epoch 9 Loss 8.2612 Time 0.01s
Epoch 10 Loss 8.2344 Time 0.01s
Epoch 11 Loss 8.2569 Time 0.01s
Epoch 12 Loss 8.1567 Time 0.01s
Epoch 13 Loss 8.1551 Time 0.01s
Epoch 14 Loss 8.1126 Time 0.01s
Epoch 15 Loss 8.0679 Time 0.01s
Epoch 16 Loss 8.0133 Time 0.01s
Epoch 17 Loss 7.9770 Time 0.01s
Epoch 18 Loss 8.0238 Time 0.01s
Epoch 19 Loss 7.9874 Time 0.01s
Epoch 20 Loss 7.9012 Time 0.01s
Epoch 21 Loss 7.9731 Time 0.01s
Epoch 22 Loss 7.9483 Time 0.01s
Epoch 23 Loss 7.8612 Time 0.01s
Epoch 24 Loss 7.8678 Time 0.01s
Epoch 25 Loss 7.8080 Time 0.01s
Epoch 26 Loss 7.8029 Time 0.01s
Epoch 27 Loss 7.7463 Time 0.01s
Epoch 28 Loss 7.7167 Time 0.01s
Epoch 29 Loss 7.7242 Time 0.01s
Epoch 30 Loss 7.6688 Time 0.01s
Epoch 31 Loss 7.6643 Time 0.01s
Epoch 32 Loss 7.6

In [190]:
# 1. Prepare inputs (length 5)
current_toy_inp = toy_inp
current_toy_tar = toy_tar[:, :-1] 

# 2. Re-create masks for the NEW length (5)
enc_padding_mask = create_padding_mask(current_toy_inp)

# Target length is now 5, so mask must be (5, 5)
look_ahead_mask = create_look_ahead_mask(tf.shape(current_toy_tar)[1])
dec_padding_mask = create_padding_mask(current_toy_tar)
combined_mask = tf.maximum(dec_padding_mask, look_ahead_mask)

# 3. Run inference
predictions = transformer(
    (current_toy_inp, current_toy_tar), 
    training=False,
    look_ahead_mask=combined_mask,
    padding_mask=enc_padding_mask
)

predicted_ids = tf.argmax(predictions, axis=-1)

print(f"Target IDs (what we wanted): {toy_tar.numpy()[0, 1:]}") 
print(f"Predicted IDs (what we got): {predicted_ids.numpy()[0]}")

Target IDs (what we wanted): [ 54  89 210   2   0]
Predicted IDs (what we got): [ 54  89 210   2   2]


In [191]:
def translate(sentence_ids):
    # sentence_ids is the English input (e.g., [12, 45, 7, 102, 0])
    encoder_input = tf.expand_dims(sentence_ids, 0)
    
    # We start the decoder with ONLY the <start> token (ID: 1)
    decoder_input = [1]
    output = tf.expand_dims(decoder_input, 0)

    for i in range(10): # Max length of 10 words
        enc_padding_mask = create_padding_mask(encoder_input)
        look_ahead_mask = create_look_ahead_mask(tf.shape(output)[1])
        dec_padding_mask = create_padding_mask(output)
        combined_mask = tf.maximum(dec_padding_mask, look_ahead_mask)

        # Predict the next word
        predictions = transformer(
            (encoder_input, output),
            training=False,
            look_ahead_mask=combined_mask,
            padding_mask=enc_padding_mask
        )

        # Select the last word from the seq_len dimension
        prediction = predictions[:, -1:, :] 
        predicted_id = tf.cast(tf.argmax(prediction, axis=-1), tf.int32)

        # If the model predicts the <end> token (ID: 2), stop!
        if predicted_id == 2:
            break

        # Otherwise, attach the predicted ID to the decoder input and continue
        output = tf.concat([output, predicted_id], axis=-1)

    return tf.squeeze(output, axis=0)

# Try it out!
result_ids = translate([12, 45, 7, 102, 0])
print(f"Final Translation IDs: {result_ids.numpy()}")

Final Translation IDs: [  1  54  89 210]


In [192]:
# Test Vocabulary (ID to Word)
# correspondance toy_inp <-> toy_tar
fr_vocab_test = {
    0: "<pad>",
    1: "<start>",
    2: "<end>",
    54: "Je",
    89: "suis",
    210: "étudiant"
}

result_ids = translate([12, 45, 7, 102, 0])

# ignore  <start> (ID 1)
decoded_words = [fr_vocab_test.get(int(idx), f"ID_{idx}") for idx in result_ids.numpy() if idx != 1]

print("-" * 30)
print(f"Generated IDs : {result_ids.numpy()}")
print(f"Final Translation : {' '.join(decoded_words)}")

------------------------------
Generated IDs : [  1  54  89 210]
Final Translation : Je suis étudiant


## Real Training with  Many Things Dataset for Translation Fr-En

### Loading and Cleaning

In [193]:
path_to_file = "../data/fra-eng/fra.txt"
lines = open(path_to_file, encoding='utf-8').read().strip().split('\n')

num_samples = 30000
text_pairs = []
for line in lines[:num_samples]:
    parts = line.split('\t')
    if len(parts) >= 2:
        eng = parts[0].lower().strip()
        fra = "[start] " + parts[1].lower().strip() + " [end]"
        text_pairs.append((eng, fra))

import random
random.shuffle(text_pairs)

### Creation of Tokenizers (Vectorization)

In [194]:
# Configuration
vocab_size = 8000
sequence_length = 20

# English Vectorization (Source)
eng_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

# French Vectorization (Target)
# We accept special characters like [ ] for our tags
fra_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1, # +1 for the offset
)

# Learn the vocabularies
train_eng_texts = [pair[0] for pair in text_pairs]
train_fra_texts = [pair[1] for pair in text_pairs]
eng_vectorization.adapt(train_eng_texts)
fra_vectorization.adapt(train_fra_texts)

### Preparation of TF Dataset

In [195]:
batch_size = 64

def format_dataset(eng, fra):
    eng = eng_vectorization(eng)
    fra = fra_vectorization(fra)
    return (eng, fra), fra # Input to the model is (eng, fra) and target is fra (shifted inside the model)

train_ds = tf.data.Dataset.from_tensor_slices((train_eng_texts, train_fra_texts))
train_ds = train_ds.batch(batch_size).map(format_dataset).shuffle(2048).prefetch(16)

# Batch example
for (batch_eng, batch_fra), label_fra in train_ds.take(1):
    print(f"Shape English : {batch_eng.shape}") # (64, 20)
    print(f"Shape French : {batch_fra.shape}") # (64, 21)

Shape English : (64, 20)
Shape French : (64, 21)


### Runing real Training session

In [197]:
epochs = 30

for epoch in range(epochs):
    train_loss.reset_state()
    
    for (batch_eng, batch_fra), label_fra in train_ds:
        # Note : Dans votre train_step, vous devrez ajuster pour prendre
        # inp = batch_eng
        # tar = batch_fra
        train_step(batch_eng, batch_fra)
    
    print(f"Epoch {epoch + 1} - Loss: {train_loss.result():.4f}")

KeyboardInterrupt: 